# Language-Table Visualization — Figure 2 for CoRL paper

Two output modes — pick one in **Cell ①** below:

| Mode | What you get | When to use |
|------|-------------|-------------|
| **Composite** (`SAVE_INDIVIDUAL = False`) | Single `lt_qual_composite.png` ready for LaTeX | Happy with current layout |
| **Individual frames** (`SAVE_INDIVIDUAL = True`) | 9 separate PNGs + `episode_info.txt` | Want to compose in Illustrator / Figma / Keynote / PowerPoint |

### Before you run
1. Open your [VERA Drive folder](https://drive.google.com/drive/folders/1BesLRj6jdzBmiQeNFmKTTWIM8ss7C8Z-) in Drive.
2. **Add shortcut to My Drive** (right-click → *Add shortcut to Drive* → *My Drive*).
3. Runtime → **Change runtime type** → **T4 GPU**.

## ① Configuration — edit here, then run all cells

In [ ]:
# ╔═══════════════════════════════════════════════════════╗
# ║  FIGURE STYLE — edit anything here freely            ║
# ╚═══════════════════════════════════════════════════════╝

# ── Output mode ───────────────────────────────────────────
SAVE_INDIVIDUAL = False   # True  → 9 separate PNGs + episode_info.txt
                          # False → one composite lt_qual_composite.png

# ── Composite layout ──────────────────────────────────────
FIG_W        = 9.5    # figure width in inches
ROW_H        = 1.90   # height per image row
TOK_H        = 0.40   # height of token annotation row
LABEL_W_FRAC = 0.22   # width fraction for 'Instruction' column

# ── Font sizes ────────────────────────────────────────────
FONT_HEADER  = 9      # 'Start / Middle / End' column headers
FONT_LABEL   = 8      # instruction text
FONT_TOKEN   = 7.2    # E_act / E_emb annotation
FONT_CAPTION = 6.5    # in-figure caption (set 0 to suppress)

# ── Resolution ────────────────────────────────────────────
COMPOSITE_DPI  = 200  # DPI for composite PNG
INDIVIDUAL_DPI = 300  # DPI for each individual frame PNG

# ── Token strings (one pair per episode row) ──────────────
# Edit these to match your actual TERA output.
TERA_TOKENS = [
    ("I pushed the object to the left.",
     "I made little progress and received a low reward."),
    ("I pushed the object upward.",
     "I made little progress and received a low reward."),
    ("I pushed the object up and to the left.",
     "I made little progress and received a low reward."),
]

print('Config loaded — SAVE_INDIVIDUAL =', SAVE_INDIVIDUAL)

## ② Install deps & mount Drive

In [ ]:
import os, sys, subprocess
from pathlib import Path

def pip_q(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_q('pyyaml', 'matplotlib', 'pillow', 'imageio')
print('Base deps OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
assert MYDRIVE.is_dir(), 'Drive mount failed'
print('Drive mounted:', MYDRIVE)

## ③ Auto-detect checkpoints and LT data on Drive

In [ ]:
import glob, random

VERA_ROOT_HINT = '/content/drive/MyDrive/VERA_LT_Real'
# Change if your folder is named differently:
# VERA_ROOT_HINT = '/content/drive/MyDrive/VERA data/VERA_LT_Real'
SEED_DIR_NAME = 'seed123'

def resolve_seed_checkpoint(vera_root, seed_name='seed123'):
    root = Path(vera_root)
    if not root.is_dir(): return None
    guesses = [
        root / 'checkpoints/lt_full_vera' / seed_name / 'best_sft_vera.pt',
        root / 'checkpoints/It_full_vera' / seed_name / 'best_sft_vera.pt',
        root / 'checkpoints/lt_full_vera' / seed_name / 'best_sft.pt',
        root / 'checkpoints/full_vera'    / seed_name / 'best_sft_vera.pt',
    ]
    for g in guesses:
        if g.is_file(): return g
    ckpt_root = root / 'checkpoints'
    if ckpt_root.is_dir():
        for cond_dir in sorted(ckpt_root.iterdir()):
            if not cond_dir.is_dir(): continue
            seed_dir = cond_dir / seed_name
            if not seed_dir.is_dir(): continue
            for name in ('best_sft_vera.pt', 'best_sft.pt', 'best.pt'):
                p = seed_dir / name
                if p.is_file(): return p
            pts = sorted(seed_dir.glob('*.pt'),
                         key=lambda p: p.stat().st_mtime, reverse=True)
            if pts: return pts[0]
    return None

def rglob_limited(root, name, max_hits=5, max_depth=8):
    hits = []
    root = Path(root).resolve()
    for dirpath, dirnames, filenames in os.walk(root):
        if len(Path(dirpath).relative_to(root).parts) > max_depth:
            dirnames[:] = []; continue
        if name in filenames:
            hits.append(Path(dirpath) / name)
            if len(hits) >= max_hits: break
    return hits

def find_lt_episode_root(root):
    root = Path(root)
    for dirpath, dirnames, _ in os.walk(root):
        if len(Path(dirpath).relative_to(root).parts) > 6: continue
        ep_dirs = [d for d in dirnames if d.startswith('episode_')]
        if not ep_dirs: continue
        if (Path(dirpath) / ep_dirs[0] / 'steps.pkl').is_file():
            return Path(dirpath)
    return None

if not Path(VERA_ROOT_HINT).is_dir():
    for candidate in MYDRIVE.rglob('VERA_LT_Real'):
        if (candidate / 'checkpoints').is_dir():
            VERA_ROOT_HINT = str(candidate)
            print('Found VERA_LT_Real at:', VERA_ROOT_HINT)
            break

SEARCH_ROOTS = [Path(VERA_ROOT_HINT)] if Path(VERA_ROOT_HINT).is_dir() else [MYDRIVE]
CKPT_NAMES   = ('best_sft_vera.pt', 'best_sft.pt', 'best.pt', 'tera_checkpoint.pt')

checkpoints, lt_roots, config_paths = [], [], []
for root in SEARCH_ROOTS:
    if not root.exists(): continue
    for name in CKPT_NAMES:
        checkpoints.extend(rglob_limited(root, name, max_hits=3))
    config_paths.extend(rglob_limited(root, 'config.yaml', max_hits=3))
    lt = find_lt_episode_root(root)
    if lt: lt_roots.append(lt)

checkpoints  = list(dict.fromkeys(checkpoints))
resolved     = resolve_seed_checkpoint(VERA_ROOT_HINT, SEED_DIR_NAME)
if resolved:
    checkpoints = [resolved] + [p for p in checkpoints if p != resolved]
lt_roots     = list(dict.fromkeys(lt_roots))
config_paths = list(dict.fromkeys(config_paths))

print('VERA root:', VERA_ROOT_HINT, '| exists:', Path(VERA_ROOT_HINT).is_dir())
print('Checkpoints:', checkpoints or '(none)')
print('LT data roots:', lt_roots or '(none)')

## ④ Setup repo

In [ ]:
import shutil

def find_file_on_drive(name, roots, max_depth=14):
    hits = []
    for root in roots:
        root = Path(root)
        if not root.is_dir(): continue
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).relative_to(root).parts) > max_depth:
                dirnames[:] = []; continue
            if name in filenames:
                hits.append(Path(dirpath) / name)
    return hits

DRIVE_SEARCH = [MYDRIVE, Path(VERA_ROOT_HINT)]
REPO = None

for hit in find_file_on_drive('colab_run_lt_simulation.py', DRIVE_SEARCH):
    if (hit.parent.parent / 'models' / 'vera_model.py').is_file():
        REPO = hit.parent.parent
        print('Found full repo on Drive:', REPO)
        break

if REPO is None:
    for folder in ['VLA-Robot-Learning', 'RLConditionedVLA']:
        p = MYDRIVE / folder
        if (p / 'models' / 'vera_model.py').is_file():
            REPO = p
            print('Found repo on Drive:', REPO)
            break

if REPO is None:
    REPO = Path('/content/RLConditionedVLA')
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/sara-kaz/RLConditionedVLA.git', str(REPO)],
                   check=True)
    print('Cloned GitHub ->', REPO)

assert (REPO / 'models' / 'vera_model.py').is_file(), 'Missing models/vera_model.py'
(REPO / 'docs').mkdir(parents=True, exist_ok=True)

sim_script = REPO / 'docs' / 'colab_run_lt_simulation.py'
if not sim_script.is_file():
    hits = find_file_on_drive('colab_run_lt_simulation.py', DRIVE_SEARCH)
    if hits:
        shutil.copy2(hits[0], sim_script)
        print('Copied sim script from Drive')
    else:
        from google.colab import files
        print('Upload colab_run_lt_simulation.py from your Mac:')
        up = files.upload()
        sim_script.write_bytes(up[next(iter(up))])

sys.path.insert(0, str(REPO))
print('Ready — REPO:', REPO)

## ⑤ Build figure from stored `steps.pkl` (fast — no simulator)
Runs in **both** composite and individual-frames mode depending on `SAVE_INDIVIDUAL` set in Cell ①.

In [ ]:
import pickle, textwrap
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Image as IPImage, display

PREFERRED = [
    'move the red star into the yellow hexagon towards the bottom center',
    'move the yellow heart to the bottom right corner',
    'place the blue cube to the top of the red circle',
]

def _extract_frame(step):
    for key in ('obs','image','rgb','pixels','frame'):
        val = step.get(key)
        if val is None: continue
        if isinstance(val, np.ndarray) and val.ndim == 3:
            return val.astype(np.uint8)
        if isinstance(val, dict):
            for sub in ('rgb','image','pixels','agentview_rgb'):
                v2 = val.get(sub)
                if v2 is not None and isinstance(v2,np.ndarray) and v2.ndim==3:
                    return v2.astype(np.uint8)
    return None

def _get_instr(steps):
    for key in ('instruction','task','lang','language_instruction'):
        val = steps[0].get(key)
        if val:
            if isinstance(val, bytes): val = val.decode()
            return str(val).lower().strip().rstrip('.')
    return ''

def load_episodes(lt_root, n=3, seed=42):
    if not lt_root or not os.path.isdir(str(lt_root)): return []
    ep_dirs = sorted(glob.glob(os.path.join(str(lt_root),'episode_*')))
    random.seed(seed); random.shuffle(ep_dirs)
    buckets = {p: None for p in PREFERRED}; extras = []
    for ep_dir in ep_dirs:
        pkl = os.path.join(ep_dir,'steps.pkl')
        if not os.path.isfile(pkl): continue
        with open(pkl,'rb') as f: steps = pickle.load(f)
        if len(steps) < 4: continue
        instr  = _get_instr(steps)
        frames = [_extract_frame(s) for s in steps]
        frames = [f for f in frames if f is not None]
        if len(frames) < 3: continue
        mid = len(frames)//2
        ep  = {'instruction':instr,'start':frames[0],'mid':frames[mid],'end':frames[-1]}
        matched = False
        for pref in PREFERRED:
            if pref in instr and buckets[pref] is None:
                buckets[pref]=ep; matched=True; break
        if not matched: extras.append(ep)
        if all(v is not None for v in buckets.values()): break
    result = []
    for pref in PREFERRED:
        result.append(buckets[pref] if buckets[pref] is not None
                      else (extras.pop(0) if extras else None))
    return [e for e in result if e is not None][:n]

PLACEHOLDER = np.full((128,128,3), 210, dtype=np.uint8)

if not lt_roots:
    print('No episode_*/steps.pkl found — using placeholder grey frames.')
    print('Run Path B (live rollout) or store LT episodes on Drive first.')
    episodes = []
else:
    LT_ROOT  = lt_roots[0]
    print('Using LT data:', LT_ROOT)
    episodes = load_episodes(LT_ROOT, n=3)
    print(f'Loaded {len(episodes)} real episodes')

for i in range(len(episodes), 3):
    episodes.append({'instruction': PREFERRED[i],
                     'start': PLACEHOLDER, 'mid': PLACEHOLDER, 'end': PLACEHOLDER})

# ─────────────────────────────────────────────────────────
# INDIVIDUAL FRAMES MODE
# ─────────────────────────────────────────────────────────
if SAVE_INDIVIDUAL:
    out_dir = Path('/content/individual_frames')
    out_dir.mkdir(parents=True, exist_ok=True)
    info_lines = ['# TERA LT individual frames', '']
    for ep_i, ep in enumerate(episodes):
        sn, ek = TERA_TOKENS[ep_i]
        info_lines += [f'## Episode {ep_i}',
                       f'Instruction : {ep["instruction"].capitalize()}.',
                       f'E_act       : {sn}',
                       f'E_emb       : {ek}', '']
        for slot, frame in [('start',ep['start']),('mid',ep['mid']),('end',ep['end'])]:
            fig_f, ax_f = plt.subplots(1,1,figsize=(3,3),facecolor='white')
            ax_f.imshow(frame); ax_f.set_xticks([]); ax_f.set_yticks([])
            for sp in ax_f.spines.values():
                sp.set_linewidth(0.8); sp.set_color('#AAAAAA')
            fig_f.tight_layout(pad=0.1)
            dest = out_dir / f'ep{ep_i}_{slot}.png'
            fig_f.savefig(str(dest), dpi=INDIVIDUAL_DPI,
                          bbox_inches='tight', facecolor='white')
            plt.close(fig_f)
            print(f'  Saved: {dest}')
    (out_dir / 'episode_info.txt').write_text('\n'.join(info_lines))

    # Copy to Drive
    drive_out = MYDRIVE / 'individual_frames'
    if drive_out.exists(): shutil.rmtree(str(drive_out))
    shutil.copytree(str(out_dir), str(drive_out))
    print(f'\n Saved 9 frames + episode_info.txt')
    print(f' Drive folder: {drive_out}')

    # Preview grid
    fig_p, axes = plt.subplots(3, 3, figsize=(9,9), facecolor='white')
    for r in range(3):
        for c, slot in enumerate(['start','mid','end']):
            p = out_dir / f'ep{r}_{slot}.png'
            if p.is_file(): axes[r][c].imshow(plt.imread(str(p)))
            axes[r][c].set_title(f'ep{r}_{slot}', fontsize=7)
            axes[r][c].axis('off')
    plt.tight_layout()
    preview_p = '/content/frames_preview.png'
    fig_p.savefig(preview_p, dpi=120, bbox_inches='tight')
    plt.close(fig_p)
    display(IPImage(filename=preview_p))

# ─────────────────────────────────────────────────────────
# COMPOSITE MODE
# ─────────────────────────────────────────────────────────
else:
    SEP_H   = 0.06
    total_h = 0.45 + 3 * (ROW_H + TOK_H + SEP_H) + 0.40
    fig     = plt.figure(figsize=(FIG_W, total_h), facecolor='white')
    outer   = gridspec.GridSpec(
        6, 1, figure=fig, hspace=0.0,
        left=0.01, right=0.99, top=0.96,
        bottom=(0.12 if FONT_CAPTION > 0 else 0.04))

    frame_axes = []
    for row_i in range(3):
        img_gs = gridspec.GridSpecFromSubplotSpec(
            1, 4, subplot_spec=outer[row_i*2],
            width_ratios=[LABEL_W_FRAC,1,1,1], wspace=0.04)
        frame_axes.append([fig.add_subplot(img_gs[j]) for j in range(4)])
        tok_gs = gridspec.GridSpecFromSubplotSpec(
            1, 1, subplot_spec=outer[row_i*2+1])
        ax_tok = fig.add_subplot(tok_gs[0]); ax_tok.axis('off')
        sn, ek = TERA_TOKENS[row_i]
        ax_tok.text(0.50, 0.80,
            f'$\\mathit{{E_{{\\mathrm{{act}}}}}}$: "{sn}"    '
            f'$\\mathit{{E_{{\\mathrm{{emb}}}}}}$: "{ek}"',
            transform=ax_tok.transAxes,
            fontsize=FONT_TOKEN, color='#2C3E50',
            va='top', ha='center', style='italic')

    COL_TITLES = ['Start','Middle','End']
    for row_i, ep in enumerate(episodes):
        ax_lbl, ax_s, ax_m, ax_e = frame_axes[row_i]
        ax_lbl.axis('off')
        wrapped = textwrap.fill(
            ep['instruction'].capitalize().rstrip('.')+'.', width=22)
        ax_lbl.text(0.95, 0.5, wrapped, transform=ax_lbl.transAxes,
                    fontsize=FONT_LABEL, ha='right', va='center', style='italic')
        for ax_img, frame, lbl in [(ax_s,ep['start'],COL_TITLES[0]),
                                    (ax_m,ep['mid'],  COL_TITLES[1]),
                                    (ax_e,ep['end'],  COL_TITLES[2])]:
            ax_img.imshow(frame); ax_img.set_xticks([]); ax_img.set_yticks([])
            for sp in ax_img.spines.values():
                sp.set_linewidth(0.6); sp.set_color('#AAAAAA')
            if row_i == 0:
                ax_img.set_title(lbl, fontsize=FONT_HEADER,
                                 fontweight='bold', pad=3)
    frame_axes[0][0].set_title('Instruction', fontsize=FONT_HEADER,
                                fontweight='bold', pad=3)

    if FONT_CAPTION > 0:
        fig.text(0.01, 0.01,
            'Figure 2. Three Language-Table demonstrations where TERA successfully '
            'follows the text instruction (start -> middle -> end frame). '
            'The per-step E_act and E_emb tokens illustrate closed-loop '
            'language feedback driving progressive trajectory correction.',
            fontsize=FONT_CAPTION, va='bottom', color='#333333', ha='left')

    out_p = Path('/content/lt_qual_composite.png')
    fig.savefig(str(out_p), dpi=COMPOSITE_DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    shutil.copy2(str(out_p), str(MYDRIVE / 'lt_qual_composite.png'))
    print(' Saved composite:', out_p)
    print(' Copied to Drive:', MYDRIVE / 'lt_qual_composite.png')
    display(IPImage(filename=str(out_p)))

## ⑥ Path B — Live TERA rollout (install simulator first, ~15–25 min on T4)
Skip this if Path A already produced good frames from stored `steps.pkl`.

In [ ]:
def pip1(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print('OK', pkg[:50])
for p in ['gym<=0.23.0', 'pybullet', 'opencv-python', 'pyyaml', 'imageio']:
    pip1(p)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'git+https://github.com/google-research/language-table.git'])
print('OK language-table')
try:
    import clip
except ImportError:
    pip1('git+https://github.com/openai/CLIP.git')
from language_table.environments import language_table as lt
print('language_table import OK')

In [ ]:
if not checkpoints:
    print('No checkpoint found — upload best_sft_vera.pt to Drive and re-run detect cell.')
else:
    CKPT = str(checkpoints[0])
    CFG  = str(config_paths[0]) if config_paths else str(REPO / 'configs' / 'config.yaml')
    print('Checkpoint:', CKPT)
    script    = (REPO / 'docs' / 'colab_run_lt_simulation.py').read_text()
    new_lines = []
    for line in script.splitlines():
        if   line.startswith('CHECKPOINT = '): line = f'CHECKPOINT = "{CKPT}"'
        elif line.startswith('CONFIG_PATH = '): line = f'CONFIG_PATH = "{CFG}"'
        elif line.startswith('OUT_PNG_REPO = '):
            line = 'OUT_PNG_REPO = "' + str(REPO/'docs'/'lt_qual_composite.png') + '"'
        elif line.startswith('OUT_PNG_SUB  = '):
            line = 'OUT_PNG_SUB  = "' + str(MYDRIVE/'lt_qual_composite.png') + '"'
        new_lines.append(line)
    patched = Path('/content/colab_run_lt_simulation_patched.py')
    patched.write_text('\n'.join(new_lines) + '\n')
    %run /content/colab_run_lt_simulation_patched.py
    out = REPO / 'docs' / 'lt_qual_composite.png'
    if out.is_file():
        display(IPImage(filename=str(out)))

## ⑦ Download to your Mac

**Composite PNG** — In Drive right-click `lt_qual_composite.png` → Download.  
Copy to:
```
~/Downloads/corl_2026_template_submission/lt_qual_composite.png
docs/lt_qual_composite.png
```

**Individual frames** (if `SAVE_INDIVIDUAL = True`) — In Drive right-click `individual_frames/` → Download.  
Contains:
```
ep0_start.png   ep0_mid.png   ep0_end.png
ep1_start.png   ep1_mid.png   ep1_end.png
ep2_start.png   ep2_mid.png   ep2_end.png
episode_info.txt   ← instruction + token strings for each row
```
Import into **Illustrator / Figma / Keynote / PowerPoint** to compose your own layout.